# 과제 10 · 12 · 전체 흐름을 잇고 ML 이 놓친 고장까지 세어 평가하기

## 현업 시나리오
부품은 다 만들었습니다. 이제 카드 한 장이 근거 수집 → 경로 → 검토 → 보고서까지 가는지 한 그래프로 확인하고, 평가해야 합니다. 그런데 Agent 만 평가하면 **ML 이 경보를 내지 않아 Agent 에게 오지 않은 고장**은 보이지 않습니다. 설비 관리자가 알고 싶은 것은 "고장 중 몇 건을 잡았나"입니다.

## 학습 목표
- 병렬 수집 · 규칙 경로 · 검토 멈춤 · 보고서를 한 그래프로 잇는다.
- 경보 카드 전체의 경로를 정답과 대조한다.
- 경보가 없던 고장까지 넣어 시스템 재현율을 계산하고, Agent 만 본 숫자와 비교한다.

## 직접 조립
완성된 App을 가져오지 않습니다. 아래에서 작은 fixture와 핵심 함수를 직접 만듭니다.

### 1단계 · 한 그래프로 잇기

In [ ]:
# 연습 카드 여섯 장입니다. met 은 판정 기준 성립 유형, failure 는 실제 고장 여부(평가 전용 정답)입니다.
# 그래프에는 failure 를 넘기지 않습니다. Agent 가 정답을 보면 평가가 성립하지 않습니다.
# 관찰 포인트: manual 과 history 가 함께 시작해 decide 로 모이고, review 에서 멈췄다가 report 로 갑니다.
import operator
from collections import Counter
from typing import Annotated, TypedDict
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt

# SOP-EA-01 6장(escalation 조건)과 9장(처리 경로)을 그대로 함수로 옮깁니다. LLM 에게 묻지 않습니다.
# 카드는 판정 결과까지 붙은 모양입니다. met 은 판정 기준이 성립한 유형입니다.
def practice_route(card):
    reasons = []
    if len(card["candidates"]) >= 2:
        reasons.append("ESC-2")
    if card["met"] and card["predicted"] not in card["met"]:
        reasons.append("ESC-3")
    if card["evidence"] == 0:
        reasons.append("ESC-4")
    if not card["met"] and card["severity"] == "alarm":
        reasons.append("ESC-7")
    if reasons:
        return "escalation", reasons
    if not card["met"]:
        return "inspect_only", ["SOP-EA-01 8"]
    return "grounded_draft", []

practice_cards = {
    "S01 HDF 정탐": {"predicted": "HDF", "candidates": ["HDF"], "met": ["HDF"], "severity": "warning", "evidence": 9},
    "S08 예측≠판정": {"predicted": "OSF", "candidates": ["OSF"], "met": ["TWF"], "severity": "alarm", "evidence": 11},
    "S05 오탐 경고": {"predicted": "HDF", "candidates": ["HDF"], "met": [], "severity": "warning", "evidence": 7},
}

evaluation_cases = [
    ({"id": "C1", "predicted": "HDF", "candidates": ["HDF"], "met": ["HDF"], "severity": "warning", "evidence": 9}, True),
    ({"id": "C2", "predicted": "OSF", "candidates": ["OSF"], "met": ["TWF"], "severity": "alarm", "evidence": 11}, True),
    ({"id": "C3", "predicted": "PWF", "candidates": ["PWF"], "met": ["PWF"], "severity": "alarm", "evidence": 8}, True),
    ({"id": "C4", "predicted": "HDF", "candidates": ["HDF"], "met": [], "severity": "warning", "evidence": 7}, False),
    ({"id": "C5", "predicted": "TWF", "candidates": ["TWF"], "met": ["TWF"], "severity": "warning", "evidence": 8}, False),
    ({"id": "C6", "predicted": "OSF", "candidates": ["OSF"], "met": ["OSF"], "severity": "alarm", "evidence": 10}, True),
]

class PracticeCase(TypedDict, total=False):
    card: dict
    evidence: Annotated[list, operator.add]
    route: str
    reasons: list
    decision: str
    report: str
    trace: Annotated[list, operator.add]

def manual(state):
    return {"evidence": [f"MC01-MM {t} 판정 기준" for t in state["card"]["met"]], "trace": ["manual"]}

def history(state):
    return {"evidence": [f"이력: 같은 예측({state['card']['predicted']})"], "trace": ["history"]}

def decide(state):
    route, reasons = practice_route(state["card"])
    return {"route": route, "reasons": reasons, "trace": [f"decide:{route}"]}

def review(state):
    answer = interrupt({"route": state["route"], "reasons": state["reasons"], "evidence": state["evidence"]})
    return {"decision": answer["decision"], "trace": [f"review:{answer['decision']}"]}

def report(state):
    text = "" if state["decision"] == "reject" else f"[{state['card']['id']}] {state['route']} {state['reasons']} · 근거 {len(state['evidence'])}건"
    return {"report": text, "trace": ["report"]}

practice_builder = StateGraph(PracticeCase)
for practice_name, practice_node in (("manual", manual), ("history", history), ("decide", decide),
                                     ("review", review), ("report", report)):
    practice_builder.add_node(practice_name, practice_node)
for practice_name in ("manual", "history"):
    practice_builder.add_edge(START, practice_name)
    practice_builder.add_edge(practice_name, "decide")
practice_builder.add_edge("decide", "review")
practice_builder.add_edge("review", "report")
practice_builder.add_edge("report", END)
practice_pipeline = practice_builder.compile(checkpointer=InMemorySaver())

practice_reports = {}
for practice_card, practice_decision in ((evaluation_cases[0][0], "approve"), (evaluation_cases[1][0], "escalate")):
    practice_config = {"configurable": {"thread_id": practice_card["id"]}}
    practice_pipeline.invoke({"card": practice_card, "trace": []}, practice_config)
    practice_out = practice_pipeline.invoke(Command(resume={"decision": practice_decision}), practice_config)
    practice_reports[practice_card["id"]] = practice_out
    print(f"{practice_card['id']}: {' → '.join(practice_out['trace'])}")
    print(f"    {practice_out['report']}")

**결과 해설** — C1 은 HDF 정탐이라 grounded_draft 로 가서 승인되었고, C2 는 예측(OSF)과 판정(TWF)이 달라 ESC-3 으로 넘겨 escalate 되었습니다. 두 건 모두 manual 과 history 가 decide 전에 돌았고, review 에서 멈췄다가 사람의 결정으로 report 까지 갔습니다.

### 2단계 · 경로를 정답과 대조

In [ ]:
# 경보 카드 전체의 경로를 정답(실제 고장인가)과 나란히 셉니다. 경로는 그래프의 decide 와 같은 규칙입니다.
# 사람 검토와 보고서는 경로에 영향을 주지 않으므로 여기서는 돌리지 않습니다.
# 관찰 포인트: inspect_only(오탐 점검)로 보낸 실제 고장이 몇 건인지가 Agent 쪽의 놓침입니다.
practice_table = Counter((practice_route(card)[0], "고장" if failure else "오탐") for card, failure in evaluation_cases)
for practice_route_name in ("grounded_draft", "escalation", "inspect_only"):
    print(f"  {practice_route_name:15} 실제 고장 {practice_table[(practice_route_name, '고장')]} · 오탐 {practice_table[(practice_route_name, '오탐')]}")
practice_agent_missed = practice_table[("inspect_only", "고장")]
practice_alarmed_failures = sum(failure for _, failure in evaluation_cases)
print(f"경보로 온 실제 고장 {practice_alarmed_failures}건 중 Agent 가 점검만 권한 것 {practice_agent_missed}건")

**결과 해설** — grounded_draft 4건 중 1건(C5)은 오탐입니다. TWF 위험 구간에 들어 기준은 성립했지만 실제 고장은 아니었습니다. inspect_only 로 보낸 실제 고장은 0건이라, Agent 만 보면 "고장을 하나도 놓치지 않았다"고 말할 수 있습니다.

### 3단계 · Agent 바깥까지 — 시스템 재현율

In [ ]:
# 같은 기간에 ML 이 경보를 내지 않은 고장입니다. 이것은 Agent 에게 오지 않았습니다.
# 시스템 재현율 = 경보로 Agent 에게 온 실제 고장 / (그것 + 경보 없이 지나간 고장)
# 관찰 포인트: Agent 만 본 숫자(100%)와 시스템 재현율이 다릅니다. 차이는 Agent 가 볼 수 없었던 고장입니다.
practice_missed = [{"at": "2025-08-03T01:30:00+09:00", "actual": ["TWF"]},
                   {"at": "2025-08-19T13:00:00+09:00", "actual": ["HDF"]}]
practice_agent_recall = (practice_alarmed_failures - practice_agent_missed) / practice_alarmed_failures
practice_system_recall = (practice_alarmed_failures - practice_agent_missed) / (practice_alarmed_failures + len(practice_missed))
print(f"Agent 만 본 재현율  {practice_agent_recall:.1%}  (경보로 온 고장 기준)")
print(f"시스템 재현율       {practice_system_recall:.1%}  (경보가 없던 고장 {len(practice_missed)}건 포함)")
print("놓친 고장의 유형:", dict(Counter(t for m in practice_missed for t in m["actual"])))

assert practice_reports["C1"]["report"].startswith("[C1] grounded_draft")
assert practice_reports["C2"]["reasons"] == ["ESC-3"]
assert practice_table == Counter({("grounded_draft", "고장"): 3, ("grounded_draft", "오탐"): 1,
                                  ("escalation", "고장"): 1, ("inspect_only", "오탐"): 1})
assert practice_agent_recall == 1.0
assert round(practice_system_recall, 3) == 0.667, "경보가 없던 고장을 평가에 넣지 않았습니다"

**결과 해설** — Agent 만 보면 재현율 100% 이지만, 경보 없이 지나간 고장 2건을 넣으면 시스템 재현율은 66.7% 입니다. 나머지는 Agent 가 아무리 잘해도 볼 수 없었던 고장이고, 그 한계는 ML 의 경보 기준이 정합니다. **Agent 만 평가하면 이 차이가 보이지 않습니다.**

## 직접 해 볼 과제
실제 앱의 평가를 돌려 보세요. 터미널에서 앱을 띄운 뒤(`uv run uvicorn task10_maintenance.app:app --port 8035`) 브라우저로 `http://127.0.0.1:8035/evaluate` 를 열거나, 화면(Streamlit)의 '평가' 탭에서 "평가 실행"을 누릅니다. 아래 셀에 그 숫자를 적고, 연습 숫자와 비교해 보세요. 경로별 오탐은 어디에 몰려 있습니까? 놓친 고장은 어떤 유형이 많습니까?

In [ ]:
# 실제 앱 /evaluate 의 숫자를 적습니다(앱을 띄우지 않았으면 None 으로 둡니다).
# 연습 데이터의 숫자와 나란히 놓고 차이를 읽어 보세요.
practice_try_app_recall = None   # 예: 0.747
print("연습 시스템 재현율:", f"{practice_system_recall:.1%}", "· 실제 앱:", practice_try_app_recall)

## 중간 결과
카드 두 장이 한 그래프에서 멈췄다가 보고서까지 가는 흐름, 경로별 실제 고장·오탐 수, 그리고 Agent 만 본 재현율과 시스템 재현율의 차이를 확인합니다.

## 실패 경계
이 평가는 합성 데이터에서 잰 숫자입니다. HDF·PWF·OSF 가 규칙만으로 정답과 잘 맞는 것은 AI4I 2020 이 이 조건식으로 고장을 만든 데이터라서이고, 실제 설비에서는 그렇지 않습니다. 또 경로가 맞았다고 초안 문장이 맞는 것은 아닙니다. 문장의 품질은 사람 검토가 봅니다.

## 실제 app 연결
과제 10 App 의 `evaluation.py::evaluate` 가 같은 평가를 실제 데이터로 합니다. 대표 사례 S01~S10 은 10건 모두 기대 경로대로 가고, 경보 카드 111장은 grounded_draft 75장(실제 고장 63) · escalation 17장(11) · inspect_only 19장(0)입니다. 운영 기간 실제 고장 99건 중 74건이 경보로 와서 **시스템 재현율은 74.7%** 이고, 놓친 25건 중 10건이 TWF 입니다. 경로는 `evaluation.py::route_for` 가 그래프와 같은 규칙으로 정하고, `GET /evaluate`(`app.py::evaluate`)가 이 결과를 돌려줍니다.

## 다음 Notebook 연결
이제 과제 10 App 전체를 읽을 준비가 되었습니다. `task10_maintenance/review.py` 의 그래프 그림부터 보고, 화면(Streamlit)에서 대표 사례 S01~S10 을 하나씩 처리해 보세요.